In [ ]:
# Librerie
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
)
from sklearn.model_selection import (
    cross_val_score,
    train_test_split,
)
from sklearn.decomposition import PCA

# High Dimensionality
from pyod.models.abod import ABOD

# Elliptic
from sklearn.covariance import EllipticEnvelope

# RKNN
from sklearn.neighbors import (
    KNeighborsClassifier,
    NearestNeighbors,
    KNeighborsRegressor,
)

# Ensemble
from pyod.models.feature_bagging import FeatureBagging
# carico LOF per FeaBag, chiaramente si può cambiare a seconda del modello semplice scelto
from pyod.models.lof import LOF
from pyod.models.loda import LODA

# Foreste
from sklearn.ensemble import IsolationForest
from isotree import IsolationForest as EIF

# Grafici
import matplotlib.pyplot as plt
import seaborn as sns

# Proiezioni
from sklearn.manifold import MDS
from sklearn.manifold import TSNE
import pacmap
import umap

# Albero
from sklearn.tree import (
    DecisionTreeClassifier,
    plot_tree)

In [ ]:
# Dati
df = pd.read_csv("cmi_internet.csv")

train_oppa = pd.read_csv("train_imputed.csv")
test_oppa = pd.read_csv("test_imputed.csv")

tt = np.concatenate([train_oppa, test_oppa], axis=0)
df_unito = pd.DataFrame(tt, columns=train_oppa.columns)

########!!!
# QUANDO USIAMO DATI VERI, GESTIRE OUTLIERS
########!!!

df_outliers = pd.read_csv("df_outliers.csv")
df_outliers["colore"] = np.where(df_outliers["risultati"] >= 2, 1, 0)

df_unito['risultati'] = df_outliers['risultati'].values
df_pulito = df_unito[df_unito['risultati'] <= 1].copy()

print(f"Righe originali: {len(df_unito)}")
print(f"Righe dopo il filtro: {len(df_pulito)}")

df_imbalanced = df_pulito.copy()
df_imbalanced = df_imbalanced.drop(["risultati", "Unnamed: 0"], axis=1)
df_imbalanced["sii"] = np.where(df_imbalanced["sii"] <= 2, 0, 1)

In [ ]:
# Scalers

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

X = df_imbalanced.drop("sii", axis=1).values
y = np.array(df_imbalanced['sii'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

In [ ]:
# Decision Tree
# cambiapre iperparametri
tree_class = DecisionTreeClassifier(
    criterion='gini', splitter="best", max_depth=7, min_samples_leaf=50, min_samples_split=0.01)
tree_class.fit(X_train, y_train)
plt.figure(figsize=(20, 4), dpi=1000)
plot_tree(tree_class, filled=True)
plt.show()

tree_class = DecisionTreeClassifier(
    criterion='gini', splitter="best", max_depth=3, min_samples_leaf=50, min_samples_split=0.01)
tree_class.fit(X_train, y_train)
plt.figure(figsize=(20, 4), dpi=1000)
plot_tree(tree_class,  filled=True)
plt.show()

y_test_pred_tree = tree_class.predict(X_test)
y_prob_tree = tree_class.predict_proba(X_test)

# Esportazione
'''tree = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_test_pred_tree
})

tree['Prob_Class_0'] = y_prob_tree[:, 0]
tree['Prob_Class_1'] = y_prob_tree[:, 1]
tree['Prob_Class_2'] = y_prob_tree[:, 2]

tree.to_csv('tree.csv', index=False)'''

In [ ]:
# 1 Definizione umbalanced task semplice e risoluzione (Decision Tree, classificazione over SII)
# 2 Sbilanciao i dati
# 3 Ri-risolviamo il task con tecniche di undermapling (1) oversampling (1)
# 4 dire quanto sono ganze le tecniche